In [1]:
import numpy as np
import pandas as pd
import nltk
import re
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from scipy.sparse import hstack, vstack
from concrete.ml.sklearn import LogisticRegression as FHELogisticRegression
import warnings
warnings.filterwarnings('ignore')

# Download NLTK data
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

STOPWORDS = set(stopwords.words('english'))

def process_text(text):
    """Text preprocessing function"""
    text = text.lower()
    text = re.sub(r'[^a-z\\s]', ' ', text)
    tokens = text.split()
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 2]
    return tokens

print("Loading data...")
# Load datasets
train_df = pd.read_csv('PubMed_20k_RCT/train.csv')
dev_df = pd.read_csv('PubMed_20k_RCT/dev.csv')
test_df = pd.read_csv('PubMed_20k_RCT/test.csv')

# Drop unnecessary columns
train_df = train_df.drop(columns=['abstract_id', 'line_id'])
dev_df = dev_df.drop(columns=['abstract_id', 'line_id'])
test_df = test_df.drop(columns=['abstract_id', 'line_id'])

print("Vectorizing text...")
# Create TF-IDF features
vectorizer = TfidfVectorizer(
    tokenizer=process_text,
    ngram_range=(1, 2),
    max_features= 5000  # Reduced from 50000 for FHE compatibility
)

# Fit on training data only
X_train_text = vectorizer.fit_transform(train_df['abstract_text'])
X_dev_text = vectorizer.transform(dev_df['abstract_text'])
X_test_text = vectorizer.transform(test_df['abstract_text'])

# Add positional features
train_rel_pos = (train_df['line_number'] / train_df['total_lines']).values.reshape(-1, 1)
dev_rel_pos = (dev_df['line_number'] / dev_df['total_lines']).values.reshape(-1, 1)
test_rel_pos = (test_df['line_number'] / test_df['total_lines']).values.reshape(-1, 1)

train_line = train_df['line_number'].values.reshape(-1, 1)
dev_line = dev_df['line_number'].values.reshape(-1, 1)
test_line = test_df['line_number'].values.reshape(-1, 1)

# Combine features
X_train = hstack([X_train_text, train_rel_pos, train_line])
X_dev = hstack([X_dev_text, dev_rel_pos, dev_line])
X_test = hstack([X_test_text, test_rel_pos, test_line])

# Convert sparse to dense for FHE (required)
print("Converting to dense format...")
X_train_dense = X_train.toarray().astype(np.float32)
X_dev_dense = X_dev.toarray().astype(np.float32)
X_test_dense = X_test.toarray().astype(np.float32)

# Encode labels
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(train_df['target'])
y_dev = label_encoder.transform(dev_df['target'])
y_test = label_encoder.transform(test_df['target'])

print(f"Training set shape: {X_train_dense.shape}")
print(f"Number of classes: {len(label_encoder.classes_)}")
print(f"Classes: {label_encoder.classes_}")

# Combine train and dev for final model
X_final_dense = np.vstack([X_train_dense, X_dev_dense])
y_final = np.concatenate([y_train, y_dev])

/home/cse23160/Downloads/Cryptomedbench/fhe_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/cse23160/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/cse23160/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/cse23160/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Loading data...
Vectorizing text...
Converting to dense format...
Training set shape: (180040, 5002)
Number of classes: 5
Classes: ['BACKGROUND' 'CONCLUSIONS' 'METHODS' 'OBJECTIVE' 'RESULTS']


In [2]:
# Step 2: Train FHE-Compatible Model
print("\n" + "="*50)
print("STEP 2: Training FHE-Compatible Model")
print("="*50)

# Train with FHE-compatible logistic regression
fhe_model = FHELogisticRegression(
    n_bits=6,  # Quantization bits (8-bit is good for FHE)
    max_iter=100,
    random_state=42
)

print("Training FHE model...")
fhe_model.fit(X_final_dense, y_final)

# Evaluate on test set
y_pred_fhe = fhe_model.predict(X_test_dense)
print("\nFHE Model Performance:")
print(classification_report(y_test, y_pred_fhe, target_names=label_encoder.classes_))


STEP 2: Training FHE-Compatible Model
Training FHE model...

FHE Model Performance:
              precision    recall  f1-score   support

  BACKGROUND       0.29      0.85      0.43      3621
 CONCLUSIONS       0.62      0.93      0.75      4571
     METHODS       0.76      0.63      0.69      9897
   OBJECTIVE       0.76      0.22      0.34      2333
     RESULTS       0.87      0.35      0.50      9713

    accuracy                           0.58     30135
   macro avg       0.66      0.60      0.54     30135
weighted avg       0.72      0.58      0.58     30135



In [3]:
# Step 3: Compile for FHE
print("\n" + "="*50)
print("STEP 3: Compiling for FHE")
print("="*50)

# Compile the model
print("Compiling model for FHE execution (this may take a few minutes)...")
fhe_circuit = fhe_model.compile(X_final_dense)

print("\nModel compiled successfully!")
print(f"Circuit statistics:")
print(f"  - Number of bits: {fhe_circuit.graph.maximum_integer_bit_width()}")
print(f"  - Complexity: {fhe_circuit.complexity}")


STEP 3: Compiling for FHE
Compiling model for FHE execution (this may take a few minutes)...

Model compiled successfully!
Circuit statistics:
  - Number of bits: 21
  - Complexity: 1447.0


In [4]:
# Step 4: Generate and Test FHE Keys
print("\n" + "="*50)
print("STEP 4: Generating FHE Keys")
print("="*50)

# Generate keys (this creates the public and private keys)
print("Generating FHE keys (this may take a few minutes)...")
fhe_model.fhe_circuit.keygen()

print("Keys generated successfully!")

# Test with a single sample
print("\nTesting FHE inference on one sample...")
sample_idx = 0
sample = X_test_dense[sample_idx:sample_idx+1]
true_label = y_test[sample_idx]

# Run inference
fhe_prediction = fhe_model.predict(sample, fhe="execute")
print(f"Sample {sample_idx}: True={label_encoder.classes_[true_label]}, FHE Pred={label_encoder.classes_[fhe_prediction[0]]}")


STEP 4: Generating FHE Keys
Generating FHE keys (this may take a few minutes)...
Keys generated successfully!

Testing FHE inference on one sample...
Sample 0: True=BACKGROUND, FHE Pred=BACKGROUND


In [5]:
# Step 5: Benchmark FHE Performance
print("\n" + "="*50)
print("STEP 5: Benchmarking FHE Performance")
print("="*50)

import time

# Test with a small batch for timing
batch_size = min(10, len(X_test_dense))
test_batch = X_test_dense[:batch_size]

# Clear timing
start_time = time.time()
fhe_predictions = fhe_model.predict(test_batch, fhe="execute")
fhe_time = time.time() - start_time

print(f"FHE Inference time for {batch_size} samples: {fhe_time:.4f} seconds")
print(f"Average per sample: {fhe_time/batch_size:.4f} seconds")

# Compare with non-FHE
start_time = time.time()
clear_predictions = fhe_model.predict(test_batch, fhe="disable")
clear_time = time.time() - start_time

print(f"Clear inference time for {batch_size} samples: {clear_time:.4f} seconds")
print(f"FHE slowdown factor: {fhe_time/clear_time:.2f}x")

# Verify accuracy matches
accuracy_match = np.array_equal(fhe_predictions, clear_predictions)
print(f"\nPredictions match between FHE and clear: {accuracy_match}")


STEP 5: Benchmarking FHE Performance
FHE Inference time for 10 samples: 4.0576 seconds
Average per sample: 0.4058 seconds
Clear inference time for 10 samples: 0.0006 seconds
FHE slowdown factor: 6974.95x

Predictions match between FHE and clear: True


In [6]:
# Step 6: Full Test Set Evaluation (Optional)
print("\n" + "="*50)
print("STEP 6: Full Test Set Evaluation (This will take time)")
print("="*50)

run_full_test = False  # Set to True if you want to run full test

if run_full_test:
    print(f"Running FHE inference on {len(X_test_dense)} samples...")
    start_time = time.time()
    
    # Process in batches to avoid memory issues
    batch_size = 10
    all_predictions = []
    
    for i in range(0, len(X_test_dense), batch_size):
        batch = X_test_dense[i:i+batch_size]
        batch_pred = fhe_model.predict(batch, fhe="execute")
        all_predictions.extend(batch_pred)
        if i % 50 == 0:
            print(f"Processed {i}/{len(X_test_dense)} samples...")
    
    total_time = time.time() - start_time
    print(f"\nTotal inference time: {total_time:.2f} seconds")
    print(f"Average per sample: {total_time/len(X_test_dense):.4f} seconds")
    
    # Calculate metrics
    from sklearn.metrics import classification_report, accuracy_score
    print("\nFHE Model Performance on Full Test Set:")
    print(classification_report(y_test, all_predictions, target_names=label_encoder.classes_))
    print(f"Accuracy: {accuracy_score(y_test, all_predictions):.4f}")
else:
    print("Skipping full test set evaluation (set run_full_test=True to enable)")


STEP 6: Full Test Set Evaluation (This will take time)
Skipping full test set evaluation (set run_full_test=True to enable)


In [7]:
# Step 7: Save and Load FHE Model
print("\n" + "="*50)
print("STEP 7: Saving and Loading FHE Model")
print("="*50)

import pickle
import os

# Save the model
model_path = "pubmed_fhe_model.pkl"
vectorizer_path = "vectorizer.pkl"
label_encoder_path = "label_encoder.pkl"

print(f"Saving model to {model_path}...")
with open(model_path, 'wb') as f:
    pickle.dump(fhe_model, f)

print(f"Saving vectorizer to {vectorizer_path}...")
with open(vectorizer_path, 'wb') as f:
    pickle.dump(vectorizer, f)

print(f"Saving label encoder to {label_encoder_path}...")
with open(label_encoder_path, 'wb') as f:
    pickle.dump(label_encoder, f)

print("All files saved successfully!")

# Example of loading and using the model
print("\n" + "="*50)
print("Loading saved model for inference example")
print("="*50)

# Load the model
with open(model_path, 'rb') as f:
    loaded_model = pickle.load(f)

# Load vectorizer and label encoder
with open(vectorizer_path, 'rb') as f:
    loaded_vectorizer = pickle.load(f)
with open(label_encoder_path, 'rb') as f:
    loaded_label_encoder = pickle.load(f)

# Test with a custom sentence
custom_sentence = "Patients were randomly assigned to receive either treatment A or placebo for 12 weeks."
print(f"\nCustom sentence: {custom_sentence}")

# Process the sentence
processed = process_text(custom_sentence)
print(f"Processed tokens: {processed}")

# Create features
text_features = loaded_vectorizer.transform([custom_sentence])
rel_pos = np.array([[0.5]])  # Assume middle of abstract
line_num = np.array([[5]])    # Assume line 5
features = hstack([text_features, rel_pos, line_num]).toarray().astype(np.float32)

# Predict
prediction = loaded_model.predict(features, fhe="disable")  # Use clear for quick demo
print(f"Predicted class: {loaded_label_encoder.classes_[prediction[0]]}")


STEP 7: Saving and Loading FHE Model
Saving model to pubmed_fhe_model.pkl...


ValueError: ctypes objects containing pointers cannot be pickled